# Nhóm 4 — Callbacks và TensorBoard cho Gender/Emotion
Notebook chỉ điều hợp runner dùng chung `src/` và `scripts/run_group4_utilities.py`. Legacy FER có EarlyStopping + ImageDataGenerator; ModelCheckpoint/ReduceLROnPlateau FER là phần mở rộng T0209, không gán ngược cho mã gốc.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/BTL03')
PROJECT_ROOT=Path('/content/BTL03')
if not (PROJECT_ROOT/'scripts/run_group4_utilities.py').is_file(): raise FileNotFoundError('Checkout BTL03 vào /content/BTL03 trước khi chạy.')
import os,sys,torch
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT))
from src.utils import seed_everything
seed_everything(42)
gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
print('GPU:',gpu)
if torch.cuda.is_available() and 'T4' not in gpu: print('Cảnh báo: runtime không phải T4 như cấu hình dự kiến.')

## Dữ liệu
Đặt WIKI arrays vào `data/raw/wiki_gender/` (NHWC; runner xoay k=3 trước NCHW) và FER CSV numeric 2311 cột vào `data/raw/fer2013/`. Validation/public-test legacy được dùng theo nguyên bản; không phải holdout độc lập.

In [ ]:
!python scripts/download_data.py check

In [ ]:
!python scripts/run_group4_utilities.py --task gender

In [ ]:
!python scripts/run_group4_utilities.py --task emotion

In [ ]:
from IPython.display import display,Image
for task in ['gender','emotion']:
    path=PROJECT_ROOT/f'outputs/figures/t0209_{task}_utilities_curves.png'
    if path.is_file(): display(Image(filename=str(path)))

In [ ]:
%load_ext tensorboard
%tensorboard --logdir experiments/tensorboard/Group4_Utilities

In [ ]:
import shutil
for relative in ['outputs/checkpoints/04_training_utilities','experiments/csv_logs','experiments/tensorboard/Group4_Utilities','outputs/figures/t0209_gender_utilities_curves.png','outputs/figures/t0209_emotion_utilities_curves.png','outputs/figures/t0209_gender_utilities_metrics.json','outputs/figures/t0209_emotion_utilities_metrics.json']:
    source=PROJECT_ROOT/relative
    if source.exists():
        target=DRIVE_ROOT/relative; target.parent.mkdir(parents=True,exist_ok=True)
        if source.is_dir(): shutil.copytree(source,target,dirs_exist_ok=True)
        else: shutil.copy2(source,target)
print('Đã đồng bộ artifacts lên Drive:',DRIVE_ROOT)